In [30]:
import pandas as pd
import numpy as np

# 1. Load the dataset
df = pd.read_csv('moa_full_division_level_without_lag.csv')

# 2. Create a proper Date column for sorting
df['date'] = pd.to_datetime(dict(year=df.year, month=df.month, day=df.day))
print(f"Loaded {len(df)} initial rows.")

Loaded 32647 initial rows.


In [31]:

# ==========================================
# STEP A: SHOW DUPLICATE DATA
# ==========================================
# Check combinations of Date, Division, and Commodity that appear more than once
duplicate_mask = df.duplicated(subset=['division', 'commodity_name', 'year', 'month', 'day'], keep=False)
duplicates_df = df[duplicate_mask].sort_values(by=['division', 'commodity_name', 'year', 'month', 'day']).reset_index(drop=True)

print(f"Total rows in dataset: {len(df)}")
print(f"Rows with overlapping (date, division, commodity): {len(duplicates_df)}")


Total rows in dataset: 32647
Rows with overlapping (date, division, commodity): 0


In [32]:

# ==========================================
# STEP B: DUPLICATE RESOLUTION LOGIC (COMMENTED OUT)
# ==========================================
'''
# IF YOU DECIDE TO FIX THE DUPLICATES, UNCOMMENT THIS BLOCK.
# Recommendation: Instead of dropping the lower/higher price, we take the AVERAGE (mean) 
# of the prices reported in that division on that specific day. This is the most standard 
# economic approach for regional pricing.
# Aggregate multiple price entries per division/commodity on a single date

df = df.groupby(
    ['date', 'year', 'month', 'day', 'division', 'commodity_name', 'retail_unit']
).agg({
    'average_price': 'mean',
    'usd_bdt_rate': 'mean'
}).reset_index()

print(f"Total clean daily records after aggregation: {len(df)}")
'''

'\n# IF YOU DECIDE TO FIX THE DUPLICATES, UNCOMMENT THIS BLOCK.\n# Recommendation: Instead of dropping the lower/higher price, we take the AVERAGE (mean) \n# of the prices reported in that division on that specific day. This is the most standard \n# economic approach for regional pricing.\n# Aggregate multiple price entries per division/commodity on a single date\n\ndf = df.groupby(\n    [\'date\', \'year\', \'month\', \'day\', \'division\', \'commodity_name\', \'retail_unit\']\n).agg({\n    \'average_price\': \'mean\',\n    \'usd_bdt_rate\': \'mean\'\n}).reset_index()\n\nprint(f"Total clean daily records after aggregation: {len(df)}")\n'

In [33]:


# ==========================================
# STEP C: CREATE LAG & ROLLING FEATURES
# ==========================================
# Sort chronologically by commodity and division
df = df.sort_values(by=['commodity_name', 'division', 'date']).reset_index(drop=True)
group = df.groupby(['commodity_name', 'division'])

# ---------------------------------------------------------------------
# A. Standard Price Lags (Capped at T-90 for 1-Year Data)
# ---------------------------------------------------------------------
df['Price_T-1']  = group['average_price'].shift(1)
df['Price_T-7']  = group['average_price'].shift(7)
df['Price_T-30'] = group['average_price'].shift(30)
df['Price_T-60'] = group['average_price'].shift(60)
df['Price_T-90'] = group['average_price'].shift(90)

# ---------------------------------------------------------------------
# B. Simple 30-Day Moving Average
# ---------------------------------------------------------------------
df['Price_30d_MA'] = group['average_price'].transform(
    lambda x: x.shift(1).rolling(30).mean()
)

# ---------------------------------------------------------------------
# C. Direct Multi-Horizon Target Variables (Fix 2)
# ---------------------------------------------------------------------
df['target_30d'] = group['average_price'].shift(-30) # Target: Price in 30 days
df['target_60d'] = group['average_price'].shift(-60) # Target: Price in 60 days
df['target_90d'] = group['average_price'].shift(-90) # Target: Price in 90 days

print("Feature engineering and multi-horizon target creation complete.")

Feature engineering and multi-horizon target creation complete.


In [34]:
# =====================================================================
# CELL 4 & 5: WEEKLY & 1-MONTH TARGETS + FEATURE ENGINEERING
# =====================================================================

# Sort chronologically by commodity and division
df = df.sort_values(by=['commodity_name', 'division', 'date']).reset_index(drop=True)
group = df.groupby(['commodity_name', 'division'])

# 1. Simple Lags & Moving Averages
df['Price_T-1']  = group['average_price'].shift(1)
df['Price_T-7']  = group['average_price'].shift(7)
df['Price_T-30'] = group['average_price'].shift(30)
df['Price_30d_MA'] = group['average_price'].transform(
    lambda x: x.shift(1).rolling(30).mean()
)

# 2. Targets for Weekly and 1-Month Horizons
df['target_7d']  = group['average_price'].shift(-7)   # 1 Week Ahead
df['target_14d'] = group['average_price'].shift(-14)  # 2 Weeks Ahead
df['target_30d'] = group['average_price'].shift(-30)  # 1 Month Ahead

# Define feature and target columns
# Updated Cell 5 feature list
feature_cols = [
    'year', 'month', 'day', 'division', 'commodity_name', 'retail_unit',
    'usd_bdt_rate', 
    'average_price',  # Today's price (T_0)
    'Price_T-1', 'Price_T-7', 'Price_T-30', 'Price_30d_MA'
]
target_cols = ['target_7d', 'target_14d', 'target_30d']

# Drop rows missing features or target_30d (loses only ~8% of the dataset at the end)
clean_single_df = df.dropna(subset=feature_cols + ['target_30d']).copy()
final_df = clean_single_df[feature_cols + target_cols]
# final_df = final_df.sort_values(by=['commodity_name', 'division', 'date']).reset_index(drop=True)
df = df.sort_values(by=['year', 'month', 'day']).reset_index(drop=True)
final_df.to_csv('moa_final_dataset_with_lag.csv', index=False)

In [35]:
from sklearn.model_selection import train_test_split
import pandas as pd

# 1. Load the finalized division-level dataset
df = pd.read_csv('moa_final_dataset_with_lag.csv')

# 2. Ensure data is strictly sorted by date for the chronological split
# This guarantees the model trains on the past and predicts the true future
df = df.sort_values(by=['year', 'month', 'day']).reset_index(drop=True)
df.to_csv('moa_final_dataset_with_lag.csv', index=False)

# ---------------------------------------------------------
# SCENARIO A: Chronological Split (True Time-Series)
# ---------------------------------------------------------

# Calculate the index for the 80% mark
split_index = int(len(df) * 0.80)

# Split the data
train_chrono = df.iloc[:split_index]
test_chrono = df.iloc[split_index:]

# Save to CSV
train_chrono.to_csv('moa_train_80_lag.csv', index=False)
test_chrono.to_csv('moa_test_20_lag.csv', index=False)

print(f"Chronological Split Complete:")
print(f"  - moa_train_80_lag.csv: {len(train_chrono)} rows (Past Data)")
print(f"  - moa_test_20_lag.csv: {len(test_chrono)} rows (Future Data)")

# ---------------------------------------------------------
# SCENARIO B: Random Split (Data Leakage Test)
# ---------------------------------------------------------

# Use train_test_split to randomly shuffle and divide the data
# random_state=42 ensures the shuffle is reproducible if you run the script again
train_random, test_random = train_test_split(df, test_size=0.20, random_state=42)

# Sort the randomly selected data chronologically before saving
train_random = train_random.sort_values(by=['year', 'month', 'day']).reset_index(drop=True)
test_random = test_random.sort_values(by=['year', 'month', 'day']).reset_index(drop=True)

# Save to CSV
train_random.to_csv('moa_train_random_80_lag.csv', index=False)
test_random.to_csv('moa_test_random_20_lag.csv', index=False)

print(f"\nRandom Split Complete:")
print(f"  - moa_train_random_80_lag.csv: {len(train_random)} rows (Mixed Timeline)")
print(f"  - moa_test_random_20_lag.csv: {len(test_random)} rows (Mixed Timeline)")

Chronological Split Complete:
  - moa_train_80_lag.csv: 20165 rows (Past Data)
  - moa_test_20_lag.csv: 5042 rows (Future Data)

Random Split Complete:
  - moa_train_random_80_lag.csv: 20165 rows (Mixed Timeline)
  - moa_test_random_20_lag.csv: 5042 rows (Mixed Timeline)
